# HMS EEG — Swin-T Spectrogram Vision Pipeline v1

Classifies harmful brain activity from 50-second EEG windows using a pretrained
Swin Transformer (Swin-T) applied to STFT-derived spectrogram images.

## Overview

This pipeline is **complementary** to the Conv1D-BiGRU raw-EEG model in
`conv1d_bigru_v1.ipynb`.  The two models see different representations of the
same signal (time-frequency image vs. raw temporal waveform), making them strong
ensemble partners.

## Section map

| Cell | Purpose |
|------|---------|
| 1 | This header |
| 2 | All imports |
| 3 | **CONFIG** — user-editable flags, sanity overrides, cfg instantiation |
| 4 | Runtime paths, seed, directory creation |
| 5 | Load `confident_train.csv`, fold split, derived columns |
| 6 | Leakage check (`assert_no_test_leakage`) |
| 7 | Sanity subsample (only runs when `SANITY_MODE = True`) |
| 8 | Build DataLoaders (EEG NPZ cache + spectrogram image cache) |
| 9 | Global prior baseline KL |
| 10 | Sanity probe: forward-pass shape check, finite-loss check |
| 11 | `run_vit_training()` — three-stage staged training |
| 12 | Plot training curves |
| 13 | OOF validation predictions + KL vs baseline |
| 14 | Generate submission CSV |
| 15 | **FINAL EVAL** (guarded, `RUN_FINAL_EVAL = False`) |

In [ ]:
# ── Cell 2: All imports ───────────────────────────────────────────────────────
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedGroupKFold

# Project utilities — shared infrastructure from the Conv1D-BiGRU pipeline.
from conv1d_bigru_utils import (
    build_confident_csvs,
    cap_rows_per_eeg,
    compute_global_prior,
    compute_prior_baseline_kl,
    ensure_dirs,
    kl_divergence_np,
    plot_history,
    prepare_cache,
    set_seed,
    setup_runtime,
)

# ViT-specific utilities.
from vit_utils import (
    SpectrogramVisionCFG,
    SwinSpectrogramClassifier,
    assert_no_test_leakage,
    build_vit_dataloaders,
    predict_vit,
    run_vit_training,
)

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Imports OK.')

In [ ]:
# ── Cell 3: CONFIG (t11) ── all user-editable flags are in this cell ─────────
# ┌─────────────────────────────────────────────────────────────────────────────┐
# │  USER-EDITABLE FLAGS                                                        │
# └─────────────────────────────────────────────────────────────────────────────┘
SANITY_MODE    = False
FOLD           = 0
CHECKPOINT_NAME = f'vit_swin_fold{FOLD}_t11.pth'
# t11: T_max=10 + wd=0.05 combined. t8 proved T_max=10 works (0.644 best val).
# t9 proved wd=0.05 doesn't kill the run (0.727). Combining both to try to
# reduce t8's train-val gap (0.489 vs 0.644) and push below 0.644.

# Backbone.
BACKBONE_NAME  = 'swin_tiny_patch4_window7_224'
IMG_SIZE       = 224

# LR / optimisation.
LR             = 2e-4       # same as t1/t8
HEAD_LR        = 1e-3
PARTIAL_UNFREEZE_STAGES = (2, 3)
WEIGHT_DECAY   = 0.05       # from t9: survivable without compile, adds regularisation
GRAD_CLIP      = 1.0
COSINE_T_MAX   = 10         # from t8: faster LR decay, peaked at epoch 8 with 0.644
BATCH_SIZE     = 64

# Epoch counts.
NUM_EPOCHS_WARMUP  = 3
NUM_EPOCHS_PARTIAL = 25
NUM_EPOCHS_FULL    = 0

# Other run control.
SEED            = 42
NUM_WORKERS     = 0
USE_AMP         = True
USE_COMPILE     = False     # MUST be False — see Bug 4 in vit_memory.md
EARLY_STOPPING  = 5
RESUME_TRAINING = True

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │  Instantiate config                                                         │
# └─────────────────────────────────────────────────────────────────────────────┘
cfg = SpectrogramVisionCFG()
cfg.fold           = FOLD
cfg.backbone_name  = BACKBONE_NAME
cfg.img_size       = IMG_SIZE
cfg.lr             = LR
cfg.head_lr        = HEAD_LR
cfg.partial_unfreeze_stages = PARTIAL_UNFREEZE_STAGES
cfg.weight_decay   = WEIGHT_DECAY
cfg.grad_clip      = GRAD_CLIP
cfg.cosine_t_max   = COSINE_T_MAX
cfg.batch_size     = BATCH_SIZE
cfg.num_epochs_warmup  = NUM_EPOCHS_WARMUP
cfg.num_epochs_partial = NUM_EPOCHS_PARTIAL
cfg.num_epochs_full    = NUM_EPOCHS_FULL
cfg.seed           = SEED
cfg.num_workers    = NUM_WORKERS
cfg.use_amp        = USE_AMP
cfg.early_stopping_patience = EARLY_STOPPING
cfg.sanity_mode    = SANITY_MODE
cfg.use_compile    = USE_COMPILE

if SANITY_MODE:
    cfg.num_epochs_warmup   = cfg.sanity_epochs_warmup
    cfg.num_epochs_partial  = cfg.sanity_epochs_partial
    cfg.num_epochs_full     = cfg.sanity_epochs_full
    cfg.batch_size          = 32
    print('SANITY MODE active: reduced epochs and sample counts.')

print(f'Checkpoint: {CHECKPOINT_NAME}')
print(f'Backbone:   {cfg.backbone_name}')
print(f'Epochs:     warmup={cfg.num_epochs_warmup}  partial={cfg.num_epochs_partial}  full={cfg.num_epochs_full}')
print(f'Stage 2 unfreeze stages: {cfg.partial_unfreeze_stages}')
print(f'Batch size: {cfg.batch_size}  |  LR: backbone={cfg.lr:.1e}  head={cfg.head_lr:.1e}')
print(f'Weight decay: {cfg.weight_decay}  |  cosine_t_max: {cfg.cosine_t_max}')
print(f'use_compile: {cfg.use_compile}  |  Resume: {RESUME_TRAINING}')

In [ ]:
# ── Cell 4: Runtime paths, seed, directory creation ──────────────────────────
runtime = setup_runtime(colab=False)

cfg.BASE_PATH   = runtime['base_path']
cfg.WORK_DIR    = runtime['work_dir']
cfg.CACHE_DIR   = runtime['cache_dir'] / 'vit'   # Independent from conv1d_bigru cache
cfg.MODELS_DIR  = runtime['models_dir']
cfg.RESULTS_DIR = runtime['results_dir']
cfg.PLOTS_DIR   = runtime['plots_dir']
cfg.num_workers = NUM_WORKERS  # use CONFIG value, not runtime default

set_seed(cfg.seed)

ensure_dirs([
    cfg.CACHE_DIR,
    cfg.MODELS_DIR,
    cfg.RESULTS_DIR,
    cfg.PLOTS_DIR,
    cfg.RESULTS_DIR / Path(CHECKPOINT_NAME).stem,
])

print('BASE_PATH  :', cfg.BASE_PATH)
print('WORK_DIR   :', cfg.WORK_DIR)
print('CACHE_DIR  :', cfg.CACHE_DIR)
print('MODELS_DIR :', cfg.MODELS_DIR)
print('RESULTS_DIR:', cfg.RESULTS_DIR)
print('Seed set to', cfg.seed)

In [ ]:

# ── Cell 5: Load confident_train.csv, fold split, derived columns ─────────────
VOTE_COLS = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

# Max sub-windows retained per unique EEG recording.
# Near-duplicate spectrogram images (overlapping 50-s windows from the same EEG)
# can cause the model to memorise recording artefacts.  n=4 was empirically
# best for CNN+BiGRU and is applied here too.
MAX_ROWS_PER_EEG = 4

conf_csv_path = cfg.BASE_PATH / 'confident_train.csv'
df_conf = pd.read_csv(conf_csv_path)
print(f'Loaded {conf_csv_path.name}: {len(df_conf)} rows')

# Path to the Kaggle-provided spectrogram parquet for each row.
# These are read by KaggleSpectrogramDataset / precompute_kaggle_spec_cache.
df_conf['spec_path'] = df_conf['spectrogram_id'].apply(
    lambda sid: cfg.BASE_PATH / 'train_spectrograms' / f'{sid}.parquet'
)

# Compute total votes and normalised soft labels.
vote_arr = df_conf[VOTE_COLS].values.astype(np.float32)
total_v   = vote_arr.sum(axis=1, keepdims=True).clip(min=1.0)
soft_arr  = vote_arr / total_v
df_conf['total_votes'] = vote_arr.sum(axis=1)
df_conf['soft_labels'] = [soft_arr[i] for i in range(len(df_conf))]

# StratifiedGroupKFold — group by patient, stratify by expert_consensus.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=cfg.seed)
df_train = df_valid = None
for fold_idx, (train_idx, valid_idx) in enumerate(
    sgkf.split(df_conf, y=df_conf['expert_consensus'], groups=df_conf['patient_id'])
):
    if fold_idx == cfg.fold:
        df_train = df_conf.iloc[train_idx].reset_index(drop=True)
        df_valid = df_conf.iloc[valid_idx].reset_index(drop=True)
        break

# Deduplicate: cap rows per EEG on the training split only.
# Validation is NOT deduplicated so OOF KL is computed on the full split.
n_before = len(df_train)
df_train = cap_rows_per_eeg(df_train, max_rows_per_eeg=MAX_ROWS_PER_EEG)
print(f'Dedup (train): {n_before} -> {len(df_train)} rows  (max {MAX_ROWS_PER_EEG} per EEG)')

print(f'\nFold {cfg.fold}:')
print(f'  Train: {len(df_train)} rows, {df_train["patient_id"].nunique()} patients')
print(f'  Valid: {len(df_valid)} rows, {df_valid["patient_id"].nunique()} patients')
print('\nTrain class distribution:')
print(df_train['expert_consensus'].value_counts().sort_index())


In [ ]:
# ── Cell 6: Leakage check ─────────────────────────────────────────────────────
# Load confident_test.csv ONLY for this assertion.
# This variable is not used in any subsequent training cell.
_conf_test_csv = cfg.BASE_PATH / 'confident_test.csv'
_df_ctest = pd.read_csv(_conf_test_csv)

assert_no_test_leakage(df_train, df_valid, _df_ctest)

# Discard the test reference — do not carry it forward.
del _df_ctest

In [ ]:
# ── Cell 7: Sanity subsample ──────────────────────────────────────────────────
# Only executes when SANITY_MODE = True.
# Uses a stratified sample so class balance is roughly preserved.
if SANITY_MODE:
    def _stratified_sample(df, n, label_col='expert_consensus', seed=42):
        n_per_class = max(1, n // df[label_col].nunique())
        sampled = (
            df.groupby(label_col, group_keys=False)
              .apply(lambda grp: grp.sample(n=min(len(grp), n_per_class), random_state=seed))
              .reset_index(drop=True)
        )
        # Top up to exactly n if rounding left us short.
        if len(sampled) < n:
            remainder = df.drop(index=sampled.index, errors='ignore')
            top_up = remainder.sample(n=min(len(remainder), n - len(sampled)), random_state=seed)
            sampled = pd.concat([sampled, top_up], ignore_index=True)
        return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

    df_train = _stratified_sample(df_train, cfg.sanity_train_samples)
    df_valid = _stratified_sample(df_valid, cfg.sanity_valid_samples)
    print(f'Sanity subset — train: {len(df_train)}, valid: {len(df_valid)}')
else:
    print('Full dataset (sanity mode off).')

In [ ]:

# ── Cell 8: Build DataLoaders ─────────────────────────────────────────────────
# Reads directly from the Kaggle-provided train_spectrograms/*.parquet files.
# No raw EEG processing or NPZ intermediate files are needed for the ViT path.
#
# What this call does:
#   1. Calls precompute_kaggle_spec_cache() to write one .npy per row to
#      cache/vit/spectrograms/train_fold{N}/ and valid_fold{N}/.
#      Already-existing files are skipped (idempotent).
#   2. Returns train/valid DataLoaders backed by KaggleSpectrogramDataset.
train_loader, valid_loader = build_vit_dataloaders(df_train, df_valid, cfg)
print(f'Train batches: {len(train_loader)}  |  Valid batches: {len(valid_loader)}')


In [ ]:

# ── Cell 8b: Spectrogram visualisation (report figure) ───────────────────────
# Shows one example per class using the Kaggle-provided spectrograms.
# Each row: one 50-s EEG window.
# Column 1: the normalised log-spectrogram (montage displayed as a single channel).
# Column 2: the same image after clip/log/z-score normalisation (what the model sees).
# Run AFTER Cell 8.
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from vit_utils import provided_spec_to_image, read_provided_spectrogram

_class_names = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']
_df_ref = df_train.reset_index(drop=True)

_samples = []
for _cls in _class_names:
    _mask = _df_ref['expert_consensus'] == _cls
    if not _mask.any():
        continue
    _row = _df_ref[_mask].iloc[0]
    try:
        _img = provided_spec_to_image(_row['spec_path'], _row['spectrogram_label_offset_seconds'], cfg)
        _samples.append((_cls, _img))
    except Exception as _e:
        print(f'  [skip] {_cls}: {_e}')

_n_rows = len(_samples)
if _n_rows == 0:
    print('No samples loaded -- check that spec_path columns are correct.')
else:
    # Two display columns: raw log-montage (R channel only) and composite RGB.
    _fig = plt.figure(figsize=(10, 2.8 * _n_rows), constrained_layout=True)
    _gs  = gridspec.GridSpec(_n_rows, 2, figure=_fig, hspace=0.05, wspace=0.05)

    for _r, (_cls, _img) in enumerate(_samples):
        # _img shape: (3, H, W) -- all 3 channels are identical (tiled log-spec).
        _mono = _img[0]                          # (H, W) single channel
        _rgb  = np.transpose(_img, (1, 2, 0))   # (H, W, 3) for imshow

        _ax0 = _fig.add_subplot(_gs[_r, 0])
        _ax0.imshow(_mono, origin='lower', aspect='auto', cmap='RdBu_r',
                    interpolation='nearest')
        if _r == 0:
            _ax0.set_title('Log-power (z-scored)', fontsize=9, fontweight='bold')
        _ax0.set_ylabel(_cls, fontsize=9, rotation=0, labelpad=55, va='center', ha='right')
        _ax0.set_xticks([]); _ax0.set_yticks([])

        _ax1 = _fig.add_subplot(_gs[_r, 1])
        # Clip z-scored values for display: map [-3, 3] to [0, 1].
        _rgb_disp = np.clip((_rgb + 3.0) / 6.0, 0, 1)
        _ax1.imshow(_rgb_disp, origin='lower', aspect='auto', interpolation='nearest')
        if _r == 0:
            _ax1.set_title('RGB input to Swin-T', fontsize=9, fontweight='bold')
        _ax1.set_xticks([]); _ax1.set_yticks([])

    _fig.suptitle(
        'Kaggle-provided STFT spectrograms fed to Swin-T\n'
        '(2x2 chain montage: LL|RL top / LP|RP bottom  --  x: time, y: freq)',
        fontsize=11, fontweight='bold',
    )
    _out_path = cfg.PLOTS_DIR / 'spectrogram_examples.png'
    _fig.savefig(_out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved to: {_out_path}')


In [ ]:
# ── Cell 9: Global prior baseline KL ─────────────────────────────────────────
prior = compute_global_prior(df_train)
baseline_kl = compute_prior_baseline_kl(df_valid, prior)

print('Global prior (train mean soft-label):')
class_names = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']
for name, p in zip(class_names, prior):
    print(f'  {name:<10} {p:.4f}')
print(f'\nBaseline KL (global prior on valid): {baseline_kl:.5f}')
print('Target: model valid KL well below this value.')

In [ ]:
# Run this once in a cell or terminal to clear the stale cache:
# import shutil
# shutil.rmtree(cfg.CACHE_DIR / 'spectrograms', ignore_errors=True)

In [ ]:
# ── Cell 10: Sanity probe ────────────────────────────────────────────────────
# Build a temporary model instance, run one batch, check shape + finite loss.
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

_probe_model = SwinSpectrogramClassifier(cfg, pretrained=False).to(device)
_probe_model.eval()

x_probe, y_probe, _ = next(iter(train_loader))
x_probe = x_probe.to(device)
y_probe = y_probe.to(device)

with torch.no_grad():
    logits_probe = _probe_model(x_probe)

assert logits_probe.shape == (x_probe.shape[0], cfg.num_classes), (
    f'Expected logits shape ({x_probe.shape[0]}, {cfg.num_classes}), '
    f'got {tuple(logits_probe.shape)}'
)

loss_probe = F.kl_div(
    F.log_softmax(logits_probe, dim=1), y_probe, reduction='batchmean'
)
assert torch.isfinite(loss_probe), f'Probe loss is not finite: {loss_probe.item()}'

n_total     = sum(p.numel() for p in _probe_model.parameters())
n_trainable = sum(p.numel() for p in _probe_model.parameters() if p.requires_grad)
print(f'Probe output shape: {tuple(logits_probe.shape)}  (expected ({x_probe.shape[0]}, {cfg.num_classes}))')
print(f'Probe KL loss: {loss_probe.item():.5f}  (finite: ok)')
print(f'Parameters — total: {n_total:,}  trainable: {n_trainable:,}')

del _probe_model, x_probe, y_probe, logits_probe
print('Sanity probe passed.')

In [ ]:
# ── Cell 11: Staged training ──────────────────────────────────────────────────
# Runs Stage 1 (head warmup), Stage 2 (partial unfreeze + CosineAnnealing +
# early stopping), and optionally Stage 3 (full fine-tune).
# Artifacts written every epoch to results/<checkpoint_stem>/.
history = run_vit_training(
    train_loader=train_loader,
    valid_loader=valid_loader,
    cfg=cfg,
    baseline_kl=baseline_kl,
    checkpoint_name=CHECKPOINT_NAME,
    resume=RESUME_TRAINING,
)
print('Training complete.')
print(f'Best valid KL: {min(history["valid_kl"]):.5f}  (baseline: {baseline_kl:.5f})')

In [ ]:
# ── Cell 12: Plot training curves ────────────────────────────────────────────
stem = Path(CHECKPOINT_NAME).stem
plot_history(
    history,
    title=f'Swin-T Spectrogram Pipeline — fold {cfg.fold}',
    cfg=cfg,
    tag=stem,
)

In [ ]:
# ── Cell 13: OOF validation predictions + KL ─────────────────────────────────
# Load best checkpoint, run inference on the full validation set.
best_ckpt_path = cfg.MODELS_DIR / (Path(CHECKPOINT_NAME).stem + '_best.pth')
ckpt = torch.load(best_ckpt_path, map_location=device)

oof_model = SwinSpectrogramClassifier(cfg, pretrained=False).to(device)
oof_model.load_state_dict(ckpt['model_state_dict'])
oof_model.eval()

oof_preds = predict_vit(oof_model, valid_loader, device, cfg)  # (N, 6)
oof_targets = np.stack(df_valid['soft_labels'].values)           # (N, 6)

oof_kl_per_sample = kl_divergence_np(oof_targets, oof_preds)
oof_kl_mean = float(oof_kl_per_sample.mean())

print(f'OOF KL  (model):    {oof_kl_mean:.5f}')
print(f'Baseline KL (prior): {baseline_kl:.5f}')
print(f'Improvement over baseline: {baseline_kl - oof_kl_mean:+.5f}')

del oof_model

In [ ]:

# ── Cell 14: Generate submission CSV ─────────────────────────────────────────
# Runs inference on the competition test set (test.csv + test_spectrograms/).
# The submission CSV is saved to results/<checkpoint_stem>/submission.csv.
from vit_utils import KaggleSpectrogramDataset, precompute_kaggle_spec_cache
from torch.utils.data import DataLoader as _DL

sub_csv_path = cfg.BASE_PATH / 'test.csv'
df_sub = pd.read_csv(sub_csv_path)

# For test rows there is no label_id, so the cache key falls back to
# spec_{spectrogram_id}_off_{offset}.
df_sub['spec_path'] = df_sub['spectrogram_id'].apply(
    lambda sid: cfg.BASE_PATH / 'test_spectrograms' / f'{sid}.parquet'
)
# test.csv uses spectrogram_label_offset_seconds; fill 0 if absent.
if 'spectrogram_label_offset_seconds' not in df_sub.columns:
    df_sub['spectrogram_label_offset_seconds'] = 0

# Precompute spectrogram image cache for the test set.
spec_test_dir = cfg.CACHE_DIR / 'spectrograms' / 'test'
precompute_kaggle_spec_cache(df_sub, spec_test_dir, cfg)

test_ds = KaggleSpectrogramDataset(
    df_sub, split='test', cfg=cfg, augment=False, precomputed_dir=spec_test_dir,
)
test_loader = _DL(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    drop_last=False,
)

# Reload the best checkpoint for inference.
sub_model = SwinSpectrogramClassifier(cfg, pretrained=False).to(device)
sub_model.load_state_dict(torch.load(best_ckpt_path, map_location=device)['model_state_dict'])
sub_model.eval()

test_preds = predict_vit(sub_model, test_loader, device, cfg)  # (N, 6)

submission = pd.DataFrame(test_preds, columns=class_names)
submission.insert(0, 'eeg_id', df_sub['eeg_id'].values)

# Save submission.
stem = Path(CHECKPOINT_NAME).stem
sub_out = cfg.RESULTS_DIR / stem / 'submission.csv'
submission.to_csv(sub_out, index=False)
print(f'Submission saved to: {sub_out}')
print(submission.head())

del sub_model


In [ ]:

# ── Cell 15: FINAL EVAL (guarded) ────────────────────────────────────────────
# This cell must NOT run during normal training.
# Set RUN_FINAL_EVAL = True only after all training and hyperparameter
# decisions are complete.  Touching confident_test.csv before that point
# invalidates the held-out evaluation.

RUN_FINAL_EVAL = False

assert RUN_FINAL_EVAL, (
    'Final evaluation is disabled. Set RUN_FINAL_EVAL = True to proceed.'
)

from vit_utils import KaggleSpectrogramDataset, precompute_kaggle_spec_cache
from torch.utils.data import DataLoader as _DL

# ── Load confident_test.csv ──────────────────────────────────────────────────
_ctest_path = cfg.BASE_PATH / 'confident_test.csv'
df_ctest = pd.read_csv(_ctest_path)

# confident_test rows come from train_spectrograms (same split as training data).
df_ctest['spec_path'] = df_ctest['spectrogram_id'].apply(
    lambda sid: cfg.BASE_PATH / 'train_spectrograms' / f'{sid}.parquet'
)

vote_arr_t = df_ctest[VOTE_COLS].values.astype(np.float32)
total_v_t  = vote_arr_t.sum(axis=1, keepdims=True).clip(min=1.0)
soft_arr_t = vote_arr_t / total_v_t
df_ctest['total_votes'] = vote_arr_t.sum(axis=1)
df_ctest['soft_labels'] = [soft_arr_t[i] for i in range(len(df_ctest))]

# Precompute spectrogram image cache for the confident test set.
spec_ctest_dir = cfg.CACHE_DIR / 'spectrograms' / 'confident_test'
precompute_kaggle_spec_cache(df_ctest, spec_ctest_dir, cfg)

ctest_ds = KaggleSpectrogramDataset(
    df_ctest, split='valid', cfg=cfg, augment=False, precomputed_dir=spec_ctest_dir,
)
ctest_loader = _DL(
    ctest_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    drop_last=False,
)

# Run inference.
eval_model = SwinSpectrogramClassifier(cfg, pretrained=False).to(device)
eval_model.load_state_dict(
    torch.load(best_ckpt_path, map_location=device)['model_state_dict']
)
eval_model.eval()

ctest_preds   = predict_vit(eval_model, ctest_loader, device, cfg)
ctest_targets = np.stack(df_ctest['soft_labels'].values)
ctest_kl      = float(kl_divergence_np(ctest_targets, ctest_preds).mean())

final_baseline_kl = compute_prior_baseline_kl(df_ctest, prior)

print(f'\n=== Final Held-Out Evaluation ===')
print(f'confident_test.csv rows: {len(df_ctest)}')
print(f'Prior baseline KL (ctest): {final_baseline_kl:.5f}')
print(f'Model KL  (ctest):         {ctest_kl:.5f}')
print(f'Improvement:               {final_baseline_kl - ctest_kl:+.5f}')

# Persist final eval artifact.
stem = Path(CHECKPOINT_NAME).stem
final_eval = {
    'checkpoint': CHECKPOINT_NAME,
    'fold': cfg.fold,
    'ctest_rows': len(df_ctest),
    'ctest_kl': ctest_kl,
    'prior_baseline_kl': final_baseline_kl,
    'improvement': float(final_baseline_kl - ctest_kl),
}
final_eval_path = cfg.RESULTS_DIR / stem / 'final_eval.json'
final_eval_path.write_text(json.dumps(final_eval, indent=2), encoding='utf-8')
print(f'\nFinal eval saved to: {final_eval_path}')

del eval_model


In [ ]:
# ── t8 confident_test evaluation ─────────────────────────────────────────────
# Runs t8's best checkpoint on confident_test.csv, saves per-sample predictions
# to the t8 results folder and the shared ensemble folder.
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader as _DL

from vit_utils import (
    KaggleSpectrogramDataset,
    SwinSpectrogramClassifier,
    SpectrogramVisionCFG,
    precompute_kaggle_spec_cache,
    predict_vit,
)
from conv1d_bigru_utils import kl_divergence_np

# ── Paths ─────────────────────────────────────────────────────────────────────
_T8_STEM       = 'vit_swin_fold0_t8'
_T8_CHECKPOINT = f'{_T8_STEM}_best.pth'
_VOTE_COLS     = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

_run_cfg_path  = Path(f'results/{_T8_STEM}/run_config_stage2.json')
_base_path     = Path('/home/littl/data/data')
_models_dir    = Path('models')
_results_dir   = Path('results')
_ensemble_dir  = Path('/home/littl/ECE247A_Final_Project/ensemble')

_confident_csv = _base_path / 'confident_test.csv'
_ckpt_path     = _models_dir / _T8_CHECKPOINT

if not _confident_csv.exists():
    raise FileNotFoundError(f'confident_test.csv not found: {_confident_csv}')
if not _ckpt_path.exists():
    raise FileNotFoundError(f't8 checkpoint not found: {_ckpt_path}')
if not _run_cfg_path.exists():
    raise FileNotFoundError(f't8 run_config not found: {_run_cfg_path}')

# ── Load run config ───────────────────────────────────────────────────────────
with open(_run_cfg_path, 'r', encoding='utf-8') as f:
    _t8_payload = json.load(f)

_jcfg        = _t8_payload['config']
_runtime     = _t8_payload['runtime']
_baseline_kl = _t8_payload['stage_metadata']['baseline_kl']  # already computed during training

def _normalize_state_dict_keys(state_dict):
    if any(k.startswith('_orig_mod.') for k in state_dict):
        return {k.replace('_orig_mod.', '', 1): v for k, v in state_dict.items()}
    return state_dict

# ── Build cfg from saved JSON ─────────────────────────────────────────────────
t8_cfg = SpectrogramVisionCFG()
for _attr in ['backbone_name', 'img_size', 'dropout', 'num_classes', 'batch_size',
              'num_workers', 'pin_memory', 'use_amp', 'use_compile',
              'n_fft', 'hop_length', 'freq_crop_hz', 'window_seconds']:
    if _attr in _jcfg:
        setattr(t8_cfg, _attr, _jcfg[_attr])

t8_cfg.BASE_PATH   = Path(_runtime['base_path'])
t8_cfg.CACHE_DIR   = Path(_runtime['cache_dir'])
t8_cfg.MODELS_DIR  = Path(_runtime['models_dir'])
t8_cfg.RESULTS_DIR = Path(_runtime['results_dir'])

print(f'Config loaded from: {_run_cfg_path}')
print(f'  backbone={t8_cfg.backbone_name}  batch={t8_cfg.batch_size}  use_compile={t8_cfg.use_compile}')

# ── Load confident_test.csv ───────────────────────────────────────────────────
confident_eval_df = pd.read_csv(_confident_csv).copy()

missing_cols = [c for c in ['eeg_id', *_VOTE_COLS] if c not in confident_eval_df.columns]
if missing_cols:
    raise ValueError(f'confident_test.csv missing columns: {missing_cols}')

confident_eval_df['spec_path'] = confident_eval_df['spectrogram_id'].apply(
    lambda sid: t8_cfg.BASE_PATH / 'train_spectrograms' / f'{sid}.parquet'
)

test_votes       = confident_eval_df[_VOTE_COLS].values.astype(np.float32)
test_vote_sums   = np.clip(test_votes.sum(axis=1, keepdims=True), 1.0, None)
test_soft_labels = test_votes / test_vote_sums
confident_eval_df['soft_labels'] = list(test_soft_labels)
confident_eval_df['total_votes'] = test_vote_sums.squeeze()

# ── Spectrogram cache + DataLoader ────────────────────────────────────────────
_spec_ct_dir = t8_cfg.CACHE_DIR / 'spectrograms' / 'confident_test'
precompute_kaggle_spec_cache(confident_eval_df, _spec_ct_dir, t8_cfg)

_ct_ds = KaggleSpectrogramDataset(
    confident_eval_df, split='valid', cfg=t8_cfg, augment=False, precomputed_dir=_spec_ct_dir,
)
_ct_loader = _DL(
    _ct_ds,
    batch_size=t8_cfg.batch_size,
    shuffle=False,
    num_workers=t8_cfg.num_workers,
    pin_memory=t8_cfg.pin_memory,
    drop_last=False,
)

# ── Load model + run inference ────────────────────────────────────────────────
_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SwinSpectrogramClassifier(t8_cfg, pretrained=False).to(_device)
ckpt  = torch.load(_ckpt_path, map_location=_device)
state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
state_dict = _normalize_state_dict_keys(state_dict)
model.load_state_dict(state_dict)
model.eval()

preds = predict_vit(model, _ct_loader, _device, t8_cfg)

if len(preds) != len(confident_eval_df):
    raise ValueError(f'Prediction length mismatch: preds={len(preds)} rows={len(confident_eval_df)}')

# ── KL divergence ─────────────────────────────────────────────────────────────
per_sample_kl = kl_divergence_np(test_soft_labels, preds)
holdout_kl    = float(per_sample_kl.mean())

# ── Build output DataFrame ────────────────────────────────────────────────────
pred_df = pd.DataFrame(preds, columns=_VOTE_COLS)
if 'label_id' in confident_eval_df.columns:
    pred_df.insert(0, 'label_id', confident_eval_df['label_id'].values)
pred_df.insert(1 if 'label_id' in pred_df.columns else 0, 'eeg_id', confident_eval_df['eeg_id'].values)
pred_df.insert(pred_df.columns.get_loc('seizure_vote'), 'per_sample_kl', per_sample_kl)

# ── Save ──────────────────────────────────────────────────────────────────────
_out_name      = f'JZ_vit_swin_confident_test_predictions_{_T8_STEM}.csv'
_local_path    = _results_dir / _T8_STEM / _out_name
_ensemble_path = _ensemble_dir / _out_name

pred_df.to_csv(_local_path,    index=False)
pred_df.to_csv(_ensemble_path, index=False)

print(f't8 checkpoint:         {_ckpt_path}')
print(f'Rows exported:         {len(pred_df)}')
print(f'Prior baseline KL:     {_baseline_kl:.5f}')
print(f't8 confident_test KL:  {holdout_kl:.5f}')
print(f'Improvement:           {_baseline_kl - holdout_kl:+.5f}')
print(f'Saved to: {_local_path}')
print(f'Saved to: {_ensemble_path}')
pred_df.head()

In [ ]:
# ── t8 confident_test confusion matrix ───────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import pandas as pd

_CLASS_NAMES = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']
_CLASS_ORDER = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

# True labels from expert_consensus; predicted from argmax of model output
_true_labels = confident_eval_df['expert_consensus'].str.strip().values
_pred_idx    = pred_df[_CLASS_ORDER].values.argmax(axis=1)
_pred_labels = [_CLASS_NAMES[i] for i in _pred_idx]

_cm      = confusion_matrix(_true_labels, _pred_labels, labels=_CLASS_NAMES)
_cm_norm = _cm.astype(float) / _cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
ConfusionMatrixDisplay(confusion_matrix=_cm, display_labels=_CLASS_NAMES).plot(
    ax=axes[0], colorbar=True, cmap='Blues', values_format='d')
axes[0].set_title('Raw counts', fontsize=11)

# Row-normalised
ConfusionMatrixDisplay(confusion_matrix=_cm_norm, display_labels=_CLASS_NAMES).plot(
    ax=axes[1], colorbar=True, cmap='Blues', values_format='.2f')
axes[1].set_title('Row-normalised (recall)', fontsize=11)

fig.suptitle(f't8 Swin-T — confident_test confusion matrix  (KL = {holdout_kl:.4f})', fontsize=13)
plt.tight_layout()

_stem = _results_dir / _T8_STEM
_cm_png  = _stem / 'confident_test_confusion_matrix.png'
_cm_csv  = _stem / 'confident_test_confusion_matrix_counts.csv'
_cmn_csv = _stem / 'confident_test_confusion_matrix_normalized.csv'

fig.savefig(_cm_png, dpi=150, bbox_inches='tight')
pd.DataFrame(_cm,      index=_CLASS_NAMES, columns=_CLASS_NAMES).to_csv(_cm_csv)
pd.DataFrame(_cm_norm, index=_CLASS_NAMES, columns=_CLASS_NAMES).to_csv(_cmn_csv)

plt.show()
print(f'Saved: {_cm_png}')
print(f'Saved: {_cm_csv}')
print(f'Saved: {_cmn_csv}')

In [ ]:
# ── Swin-T architecture diagram ───────────────────────────────────────────────
# Simplified 5-box layout: Input → Swin-T Backbone → Norm+Pool → Head → Output
# Colors match the CNN+BiGRU diagram by layer type.
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import shutil

_T8_RESULTS = cfg.RESULTS_DIR / 'vit_swin_fold0_t8'
_T8_RESULTS.mkdir(parents=True, exist_ok=True)
OUT_PATH = cfg.PLOTS_DIR / 'vit_swin_architecture_diagram.png'

# ── colours — identical to Conv1D-BiGRU diagram by role ──────────────────────
C = dict(
    input ='#D6EAF8',   # light blue – raw EEG input
    spec  ='#5DADE2',   # darker blue – spectrogram input
    conv  ='#A9DFBF',   # green  – patch embed
    gru   ='#FAD7A0',   # orange – Swin backbone
    attn  ='#D7BDE2',   # purple – Norm + Pooling
    head  ='#F9E79F',   # yellow – classification head
    out   ='#F1948A',   # red    – output
)

# ── figure setup ──────────────────────────────────────────────────────────────
FW, FH = 15, 2.9
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# ── geometry ──────────────────────────────────────────────────────────────────
N_BOXES = 5
BW   = 0.130
BH   = 0.40
GAP  = 0.040
CY   = 0.65   # raised — no brackets above boxes

MARGIN = (1.0 - N_BOXES * BW - (N_BOXES - 1) * GAP) / 2
CXS    = [MARGIN + BW / 2 + i * (BW + GAP) for i in range(N_BOXES)]

SHAPE_Y  = CY - BH / 2 - 0.065
LEGEND_Y = SHAPE_Y - 0.12

# ── helpers ───────────────────────────────────────────────────────────────────
def _box(i, lines, color='white'):
    cx = CXS[i]
    lx, ly = cx - BW / 2, CY - BH / 2
    ax.add_patch(FancyBboxPatch(
        (lx, ly), BW, BH,
        boxstyle='round,pad=0.012',
        facecolor=color, edgecolor='#555555', linewidth=1.2,
        zorder=2,
    ))
    n = len(lines)
    for j, txt in enumerate(lines):
        rel = (j + 0.5) / n
        y   = CY + BH * (0.5 - rel) * 0.78
        fs  = 12.0 if j == 0 else 10.5
        ax.text(cx, y, txt, ha='center', va='center',
                fontsize=fs, fontweight='bold' if j == 0 else 'normal',
                color='#111111' if j == 0 else '#333333', zorder=3)

def _arrow(i_from, i_to, shape_txt=None):
    inset = GAP * 0.15
    x0  = CXS[i_from] + BW / 2 + inset
    x1  = CXS[i_to]   - BW / 2 - inset
    mid = (x0 + x1) / 2
    ax.annotate('', xy=(x1, CY), xytext=(x0, CY),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3), zorder=1)
    if shape_txt:
        ax.text(mid, SHAPE_Y, shape_txt, ha='center', va='center',
                fontsize=9.0, color='#666666', style='italic', zorder=3)

# ── blocks ────────────────────────────────────────────────────────────────────
_box(0, ['Spectrogram',     '3 channels',       '224 × 224',        '(float32)'],      C['spec'])
_box(1, ['Swin-T Backbone', 'patch embed',       '+ 4 Swin stages',  'ImageNet init'],  C['gru'])
_box(2, ['LayerNorm',       'GlobalAvg',         'Pool',             '(768,)'],         C['attn'])
_box(3, ['Head',            '768 → 256',         'SiLU  Drop(0.3)',  '256 → 6'],        C['head'])
_box(4, ['Softmax',         'Output',            '6-class',          'prob. dist.'],    C['out'])

# ── arrows with tensor shapes ─────────────────────────────────────────────────
_arrow(0, 1, '(3, 224, 224)')
_arrow(1, 2, '(7×7, 768ch)')
_arrow(2, 3, '(768,)')
_arrow(3, 4, '(6,)')

# ── legend ────────────────────────────────────────────────────────────────────
ax.legend(handles=[
    mpatches.Patch(facecolor=C['spec'],  edgecolor='#555555', label='Spectrogram input'),
    mpatches.Patch(facecolor=C['gru'],   edgecolor='#555555', label='Swin-T backbone'),
    mpatches.Patch(facecolor=C['attn'],  edgecolor='#555555', label='Norm + Pooling'),
    mpatches.Patch(facecolor=C['head'],  edgecolor='#555555', label='Classification head'),
    mpatches.Patch(facecolor=C['out'],   edgecolor='#555555', label='Output'),
], loc='lower center', fontsize=9.5, framealpha=0.0, edgecolor='none',
   bbox_to_anchor=(0.5, LEGEND_Y),
   ncol=5, columnspacing=0.7, handlelength=1.2, handletextpad=0.4, borderpad=0.2)

# ── title ─────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975,
        'Kaggle Spectrogram  →  Swin-T Backbone  →  Classification Head  '
        '→  6-class probability distribution',
        ha='center', va='center', fontsize=13.0, fontweight='bold')

fig.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
shutil.copy(OUT_PATH, _T8_RESULTS / 'vit_swin_architecture_diagram.png')

print(f'Saved: {OUT_PATH}')
print(f'Copied to: {_T8_RESULTS / "vit_swin_architecture_diagram.png"}')
plt.show()
plt.close(fig)
